# C4 3D simulation of the axial fan

## Introduction

In this session we are going to create the 3D model of a blade and simulate the corresponding circular sector of the axial fan. Simulating only the sector 
is a usual practice in Turbomachines CFD, as we saw in the session for the 2D cascade, since it increases the space resolution using the cyclic symmetry of
the model. 

![sector](./images/C4_sector.png) 

Image from [CFD SUPPORT](https://www.cfdsupport.com/axial-fan-design-and-simulation/)

However, [Simscale doesn't support](https://www.simscale.com/docs/simulation-setup/boundary-conditions/periodic-boundary-condition/) this type of boundary condition for CFD so far. Hence, the complete 3D model will be simulated, using a _rotating zone_, and main results will be displayed.

## Learning objectives

1. **one**
2. **two**

## Previous tasks



## Generation of the 3D model of the axial fan blade

We are going to keep using the NACA 65(1)-212 airfoil, uncambered, considering that the stagger angle and the chord length are these:
- hub: $r = 32 mm$, $\xi_h = 20^\circ$, $l_h = 15 mm$ 
- rms: $r = 90 mm$, $\xi_h = 50^\circ$, $l_h = 30 mm$
- tip: $r = 125 mm$, $\xi_h = 70^\circ$, $l_h = 50 mm$

The airfoil is imported as CSV and loaded with `Routing curve` as explained in [`C2`](./C2_boundary_layer_and_GCI.ipynb) notebook. But, we fill it and move it 0.5 m in the $x$ direction to center it in the origin of coordinates. The sharp shape of the trailing edge will give problems when meshing, so it is an usual practice to [cut off](https://www.reddit.com/r/CFD/comments/144ai55/how_to_mesh_airfoil_trailing_edge/) a small portion of the airfoil (about 0.5 to 1% of the chord). 

This surface will serve as template for the airfoils in the three sections.

The main tool to create the blade is the **loft**. It uses two or more elements to create a surface or a solid that smoothly connect all of them. When the blade is made, we created the fluid zone (plenum and duct between hub and shroud), copy  the other 4 blades and substract the blades to get the final computational domain.

1. Import the CSV file with the airfoil data, fill it to create a surface and move it to center it in the origin of coordinates. Create a plane nomral to $x$ axis, locate at $x = 0.47\,\text{mm}$ (that is about 3% of the chord, but we want a _good_ mesh...) and split the surface, keeping only the big part.
   ![Template airfoil](./images/C4_template_airfoil.png)
2. Scale it with a factor of 0.015, with the origin of coordinates as reference point, and check that the `Copy part` box is enabled.
   ![scale](./images/C4_scale.png)
3. Translate the surface 31.7 mm in the $z$ direction and rotate it $20^\circ$ along the $z$ axis. Use the Mate connector associated with the $x-y$ plane, or create and edge with a sketch along the $z$ axis. The Mate Connector can be rotated to align with the $z$ direction, if needed. The hub is 32 mm of diameter, but the first blade section is located 0.3 mm to avoid small gaps due to the curvature of the hub. 
   
   ![Translation](./images/C4_translation.png)
   ![Rotation](./images/C4_rotation.png)
4. Repeat this operation, but with no need of modifying the location, with the other two sections.
5. Make a **loft** and a `Part 1` with the blade will be created in the `Part Studio`.
   ![blade](./images/C4_blade.png)
6. The final step is to _trim_ the pitch of the blade to fit the circular shape of the shroud. Create a sketch in the _Right_ plane with an arc like shown in the picture
   
   ![trim sketch](./images/C4_trim_sketch.png)
   
   and make an extrusion of this sketch that covers all the blade
   
   ![trim extrusion](./images/C4_trim_extrusion.png)

   Finally, split the part, keeping only the blade part.

   ![trim split](./images/C4_trim_split.png)

   and the blade is done!

   The final step is to copy this blade 4 more times, with the _circular pattern_ tool. It will create 4 more parts.

   ![cirular pattern](./images/C4_5_blades.png)

7. Let's now make the fluid domain. Create the sketch shown in the next picture on the _front_ plane.

   ![fluid sketch](./images/C4_sketch_fluid.png)

   and revolve it to create the 3D fluid domain

   ![fluid](./images/C4_fluid.png)
   ![fluid](./images/C4_fluid_section.png)

8. The last step is to substract the 5 blades from the fluid volume and the computational domain is finished. It is convenient to insert it in the Assembly tab in order to import it properly in Simscale

   ![axial fan geometry](./images/C4_axial_fan_geometry.png) 





## Simulation in Simscale

1. Like in the [C2](./C2_boundary_layer_and_GCI.ipynb) notebook, in Simscale the geometry can be directly imported from onShape, from the `Part Studio` or from the `Assembly`. After importing the geomtry, a cylinder has to be generated in the CAD app of Simscale, in order to use it as rotating zone (see [C1](./C1_simscale_and_first_simulation.md) notebook). This cylinder has to enclose the whole rotor
   
   ![cylinder](./images/C4_CFD_cylinder.png)

2. Define the material `Air` in the fluid domain.
3. Define the boundary conditions. The big circular face will be `Pressure inlet` with total pressure 0. The small circle is `Pressure outlet` with static pressure 0. Note that in this case the flow will be _pressure driven_ and the output of the simulation will be the flow rate impulsed by the fan. Define the walls for the plenum, shroud, hub and rotor.
4. Define the MRF rotating zone
   
   ![cylinder](./images/C4_CFD_MRF.png)

5. Make the mesh. In the present case, since the geometry is more complex and there are small gaps, it is better to use the `standard mesher`. Use for now a `Fineness` of 4
   
   ![mesh](./images/C4_CFD_Mesh.png)



6. Open the `Numerics` section. For high non-orthogonal meshes, like this one, it is recommended to increase the number of non-orthogonal corrector loops to 2 or 3, and switch the relaxation type to automatic.

![numerics](./images/C4_numerics.png)

